# J10f — Éducation en Île-de-France

## Objectifs
- Extraire les observations du domaine C de la BPE 2025.
- Conserver le détail des établissements, sections et services.
- Calculer les nombres par groupe et sous-catégorie.
- Calculer les parts des groupes et les parts intragroupe.
- Calculer les densités uniquement par groupe.
- Enrichir le profil communal J10e.

## Unité de comptage
Une observation BPE, pas nécessairement un campus ou un bâtiment distinct.

## Effectifs
Les effectifs disponibles sont conservés dans la table détaillée
avec leur période de référence.
Aucun total général d'élèves ou d'étudiants n'est calculé.

## Limites
- Présence dans la source distincte d'une ouverture actuelle vérifiée.
- Nombre d'observations distinct du nombre d'élèves et d'étudiants.
- Population scolarisée sur place distincte de la population résidente.
- Aucune mesure de fréquentation quotidienne.

In [1]:
#Import librairies


import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)

Pandas : 2.2.2


In [2]:
#Définir les chemins

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_INTERIM = RACINE / "data" / "interim" / "j10f"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

FICHIER_BPE = (
    RACINE / "data" / "raw" / "bpe" / "BPE25.csv"
)

FICHIER_PROFIL = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10e.csv"
)

FICHIER_METADATA_J10E = (
    RACINE / "data" / "interim" / "j10e" / "metadata_j10e.json"
)

FICHIER_DETAIL = (
    DOSSIER_PROCESSED / "education_idf_j10f.csv"
)

FICHIER_COMMUNES = (
    DOSSIER_INTERIM / "education_communes.csv"
)

FICHIER_SORTIE = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10f.csv"
)

for fichier in [FICHIER_BPE, FICHIER_PROFIL]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Profil d'entrée :", FICHIER_PROFIL)
print("Profil de sortie :", FICHIER_SORTIE)

Profil d'entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10e.csv
Profil de sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10f.csv


In [3]:
#Fonctions utiles

def normaliser_nom_colonne(nom):
    nom = unicodedata.normalize(
        "NFKD", str(nom).strip().upper()
    )
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^A-Z0-9]+", "_", nom).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    manquants = {
        "", "NA", "NAN", "NONE", "NULL",
        "<NA>", "_U", "_Z", "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(manquants)
    )


def normaliser_code_commune(serie):
    resultat = nettoyer_texte(serie).str.upper()
    resultat = resultat.str.replace(r"\.0$", "", regex=True)

    masque = resultat.str.fullmatch(r"\d{1,5}", na=False)

    return resultat.where(
        ~masque,
        resultat.str.zfill(5),
    )


def convertir_nombre(serie):
    return pd.to_numeric(
        nettoyer_texte(serie)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def sauvegarder_csv(table, chemin):
    table.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig",
    )


def sha256_fichier(chemin):
    empreinte = hashlib.sha256()

    with open(chemin, "rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    return empreinte.hexdigest()

In [4]:
#Charger le profil J10e

profil = pd.read_csv(
    FICHIER_PROFIL,
    dtype="string",
    encoding="utf-8-sig",
)

profil.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil.columns
]

if profil.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans le profil.")

if "CODGEO" not in profil.columns:
    raise ValueError("La colonne CODGEO est absente.")

profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

if not profil["CODGEO"].str.fullmatch(r"\d{5}", na=False).all():
    raise ValueError("Codes communaux manquants ou invalides.")

if not profil["CODGEO"].is_unique:
    raise ValueError("Plusieurs lignes existent pour une commune.")

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91", "92", "93", "94", "95"
}

if not profil["CODGEO"].str[:2].isin(DEPARTEMENTS_IDF).all():
    raise ValueError("Le profil contient des codes hors IDF.")

if "NB_OBS_SPORTS_LOISIRS_CULTURE_BPE2025" not in profil.columns:
    raise ValueError("Le total attendu du J10e est absent.")

codes_profil = set(profil["CODGEO"])

print("Communes :", len(profil))
print("Colonnes initiales :", len(profil.columns))

display(profil.head())

Communes : 1266
Colonnes initiales : 871


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,NB_OBS_LOISIRS_PLEIN_AIR_BPE2025,NB_OBS_CULTURE_BPE2025,NB_OBS_SPORTS_LOISIRS_CULTURE_BPE2025,PART_SPORTS_J10E_PCT,PART_LOISIRS_PLEIN_AIR_J10E_PCT,PART_CULTURE_J10E_PCT,PART_NATATION_DANS_GROUPE_PCT,PART_BOULES_PETANQUE_DANS_GROUPE_PCT,PART_TENNIS_DANS_GROUPE_PCT,PART_SKI_DANS_GROUPE_PCT,PART_EQUITATION_DANS_GROUPE_PCT,PART_ATHLETISME_DANS_GROUPE_PCT,PART_GOLF_DANS_GROUPE_PCT,PART_PARCOURS_SPORT_SANTE_DANS_GROUPE_PCT,PART_SPORTS_GLACE_DANS_GROUPE_PCT,PART_JEUX_SPORTIFS_EXTERIEURS_DANS_GROUPE_PCT,PART_TERRAINS_GRANDS_SPORTS_DANS_GROUPE_PCT,PART_SPORTS_COMBAT_DANS_GROUPE_PCT,PART_SALLES_SPORTIVES_NON_SPECIALISEES_DANS_GROUPE_PCT,PART_SPORTS_NAUTIQUES_DANS_GROUPE_PCT,PART_BOWLING_DANS_GROUPE_PCT,PART_FITNESS_REMISE_EN_FORME_DANS_GROUPE_PCT,PART_GYMNASES_MULTISPORTS_DANS_GROUPE_PCT,PART_SPORTS_MECANIQUES_DANS_GROUPE_PCT,PART_MURS_FRONTONS_DANS_GROUPE_PCT,PART_TIR_DANS_GROUPE_PCT,PART_SPORTS_AERIENS_DANS_GROUPE_PCT,PART_MODELISME_DANS_GROUPE_PCT,PART_ESCALADE_ARTIFICIELLE_DANS_GROUPE_PCT,PART_CYCLISME_DANS_GROUPE_PCT,PART_SALLES_SPORTIVES_SPECIALISEES_DANS_GROUPE_PCT,PART_SKATE_ROLLER_VELO_FREESTYLE_DANS_GROUPE_PCT,PART_BAIGNADE_AMENAGEE_DANS_GROUPE_PCT,PART_PLAISANCE_MOUILLAGE_DANS_GROUPE_PCT,PART_RANDONNEE_ORIENTATION_DANS_GROUPE_PCT,PART_SPORTS_NATURE_DANS_GROUPE_PCT,PART_CINEMAS_DANS_GROUPE_PCT,PART_CONSERVATOIRES_DANS_GROUPE_PCT,PART_BIBLIOTHEQUES_DANS_GROUPE_PCT,PART_EXPOSITION_MEDIATION_CULTURELLE_DANS_GROUPE_PCT,PART_PATRIMOINE_ESPACES_REMARQUABLES_DANS_GROUPE_PCT,PART_ARCHIVES_DANS_GROUPE_PCT,PART_ARTS_SPECTACLE_DANS_GROUPE_PCT,DENSITE_OBS_SPORTS_10000_HAB,DENSITE_OBS_LOISIRS_PLEIN_AIR_10000_HAB,DENSITE_OBS_CULTURE_10000_HAB,NB_OBS_CINEMAS_CAPACITE_CONNUE,FAUTEUILS_CINEMA_SOMME_CONNUE,FAUTEUILS_CINEMA_TOTAL_BPE2025,STATUT_CAPACITE_CINEMA
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.414592524,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.93055085738077,39.40474191053151,59.38860910108081,39.8307111089857,60.03060590120033,200.61215141394868,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,0,402,2034,80.23598820058997,0.0,19.76401179941003,5.147058823529411,3.1862745098039214,4.779411764705882,0.0,0.6740196078431373,4.901960784313726,0.30637254901960786,0.49019607843137253,0.12254901960784313,14.950980392156863,2.8799019607843137,8.14950980392157,5.392156862745098,0.12254901960784313,0.3676470588235294,14.950980392156863,16.360294117647058,0.0,1.3480392156862746,0.7965686274509803,0.061274509803921566,0.0,2.267156862745098,0.12254901960784313,11.887254901960784,0.7352941176470588,<NA>,<NA>,<NA>,<NA>,19.154228855721392,4.228855721393035,17.16417910447761,20.149253731343283,4.228855721393035,0.

In [5]:
#Définir les groupes et sous-catégories

GROUPES = {
    "C1": "ENSEIGNEMENT_PREMIER_DEGRE",
    "C2": "ENSEIGNEMENT_COLLEGES",
    "C3": "ENSEIGNEMENT_LYCEES_SECTIONS",
    "C4": "SUPERIEUR_NON_UNIVERSITAIRE",
    "C5": "SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE",
    "C6": "FORMATION_CONTINUE_APPRENTISSAGE",
    "C7": "SERVICES_UNIVERSITAIRES",
}

TYPES = {
    # Premier degré
    "C107": "ECOLES_MATERNELLES",
    "C108": "ECOLES_MIXTES_MATERNELLE_ELEMENTAIRE",
    "C109": "ECOLES_ELEMENTAIRES",

    # Collèges
    "C201": "COLLEGES",

    # Lycées et sections
    "C301": "LYCEES_GENERAUX_TECHNOLOGIQUES",
    "C302": "LYCEES_PROFESSIONNELS",
    "C303": "LYCEES_ET_ETABLISSEMENTS_SECONDAIRES_AGRICOLES",
    "C304": "SECTIONS_GENERALES_TECHNOLOGIQUES",
    "C305": "SECTIONS_PROFESSIONNELLES",

    # Supérieur non universitaire
    "C401": "FORMATIONS_STS_CPGE",
    "C403": "FORMATIONS_COMMERCE_GESTION",
    "C409": "AUTRES_FORMATIONS_POSTBAC_NON_UNIVERSITAIRES",
    "C410": "ECOLES_FORMATIONS_SANITAIRES_SOCIALES",
    "C411": "ECOLES_METIERS_CULTURE",

    # Supérieur universitaire et assimilé
    "C501": "UFR_COMPOSANTES",
    "C502": "INSTITUTS_UNIVERSITAIRES",
    "C503": "ECOLES_INGENIEURS",
    "C504": "ENSEIGNEMENT_SUPERIEUR_GENERAL_PRIVE",
    "C505": "ENSEIGNEMENT_SUPERIEUR_AGRICOLE",
    "C509": "AUTRES_STRUCTURES_ENSEIGNEMENT_SUPERIEUR",

    # Formation continue et apprentissage
    "C602": "GRETA",
    "C603": "FORMATION_CONTINUE_AGRICOLE",
    "C604": "FORMATIONS_METIERS_SPORT",
    "C610": "ORGANISMES_FORMATION_APPRENTISSAGE",

    # Services universitaires
    "C701": "RESIDENCES_UNIVERSITAIRES",
    "C702": "RESTAURANTS_UNIVERSITAIRES",
}

assert len(TYPES.values()) == len(set(TYPES.values()))
assert not (set(TYPES.values()) & set(GROUPES.values()))

referentiel = pd.DataFrame({
    "TYPEQU": list(TYPES),
    "CATEGORIE_ANALYSE": list(TYPES.values()),
})

referentiel["GROUPE_CODE"] = referentiel["TYPEQU"].str[:2]
referentiel["GROUPE_ANALYSE"] = (
    referentiel["GROUPE_CODE"].map(GROUPES)
)

sauvegarder_csv(
    referentiel,
    DOSSIER_INTERIM / "referentiel_education.csv",
)

display(referentiel)

,TYPEQU,CATEGORIE_ANALYSE,GROUPE_CODE,GROUPE_ANALYSE
0,C107,ECOLES_MATERNELLES,C1,ENSEIGNEMENT_PREMIER_DEGRE
1,C108,ECOLES_MIXTES_MATERNELLE_ELEMENTAIRE,C1,ENSEIGNEMENT_PREMIER_DEGRE
2,C109,ECOLES_ELEMENTAIRES,C1,ENSEIGNEMENT_PREMIER_DEGRE
3,C201,COLLEGES,C2,ENSEIGNEMENT_COLLEGES
4,C301,LYCEES_GENERAUX_TECHNOLOGIQUES,C3,ENSEIGNEMENT_LYCEES_SECTIONS
5,C302,LYCEES_PROFESSIONNELS,C3,ENSEIGNEMENT_LYCEES_SECTIONS
6,C303,LYCEES_ET_ETABLISSEMENTS_SECONDAIRES_AGRICOLES,C3,ENSEIGNEMENT_LYCEES_SECTIONS
7,C304,SECTIONS_GENERALES_TECHNOLOGIQUES,C3,ENSEIGNEMENT_LYCEES_SECTIONS
8,C305,SECTIONS_PROFESSIONNELLES,C3,ENSEIGNEMENT_LYCEES_SECTIONS
9,C401,FORMATIONS_STS_CPGE,C4,SUPERIEUR_NON_UNIVERSITAIRE


In [6]:
#En-tete du fichier BPE

with open(
    FICHIER_BPE,
    "r",
    encoding="utf-8-sig",
) as flux:
    premiere_ligne = flux.readline()

separateurs_valides = []

for separateur in [";", ",", "\t", "|"]:
    colonnes = next(
        csv.reader([premiere_ligne], delimiter=separateur)
    )

    colonnes = [
        normaliser_nom_colonne(colonne)
        for colonne in colonnes
    ]

    if {"AN", "DEPCOM", "TYPEQU"}.issubset(colonnes):
        separateurs_valides.append(separateur)

if len(separateurs_valides) != 1:
    raise ValueError(
        "En-tête BPE détaillé non reconnu. "
        "AN, DEPCOM et TYPEQU sont attendus."
    )

SEPARATEUR_BPE = separateurs_valides[0]

entete = pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding="utf-8-sig",
    nrows=0,
)

noms = [
    normaliser_nom_colonne(colonne)
    for colonne in entete.columns
]

if len(noms) != len(set(noms)):
    raise ValueError("Colonnes ambiguës après normalisation.")

print("Séparateur :", repr(SEPARATEUR_BPE))

Séparateur : ';'


In [7]:
#Extraire les observations par blocs

blocs_retenus = []
nombre_lignes_lues = 0
codes_inconnus = set()

with pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    dtype="string",
    encoding="utf-8-sig",
    keep_default_na=False,
    chunksize=20_000,
) as lecteur:

    for numero_bloc, bloc in enumerate(lecteur, start=1):
        bloc.columns = [
            normaliser_nom_colonne(c) for c in bloc.columns
        ]

        bloc["RANG_SOURCE_J10F"] = np.arange(
            nombre_lignes_lues + 1,
            nombre_lignes_lues + len(bloc) + 1,
        )

        nombre_lignes_lues += len(bloc)

        types = nettoyer_texte(bloc["TYPEQU"]).str.upper()
        communes = normaliser_code_commune(bloc["DEPCOM"])

        est_idf = communes.str[:2].isin(DEPARTEMENTS_IDF)
        est_education = types.str.startswith("C", na=False)

        inconnus = est_idf & est_education & ~types.isin(TYPES)

        codes_inconnus.update(
            types.loc[inconnus].dropna().tolist()
        )

        masque = est_idf & types.isin(TYPES)
        selection = bloc.loc[masque].copy()

        if not selection.empty:
            selection["TYPEQU_ANALYSE"] = types.loc[masque]
            selection["CODGEO_SOURCE"] = communes.loc[masque]
            blocs_retenus.append(selection)

        if numero_bloc % 25 == 0:
            print(f"{nombre_lignes_lues:,} lignes examinées")

if codes_inconnus:
    raise ValueError(
        "Codes du domaine C à vérifier : "
        + ", ".join(sorted(codes_inconnus))
    )

if not blocs_retenus:
    raise ValueError("Aucune observation retenue.")

education = pd.concat(blocs_retenus, ignore_index=True)
del blocs_retenus

if not nettoyer_texte(education["AN"]).eq("2025").fillna(False).all():
    raise ValueError("Millésime différent de 2025 dans l'extraction.")

print("Observations retenues :", len(education))

500,000 lignes examinées
1,000,000 lignes examinées
2,000,000 lignes examinées
2,500,000 lignes examinées
Observations retenues : 12726


In [8]:
#Tracabilité et classification

HASH_BPE = sha256_fichier(FICHIER_BPE)
DATE_TRAITEMENT = datetime.now(timezone.utc).isoformat()

education["ID_OBSERVATION"] = (
    "BPE2025_"
    + HASH_BPE[:16]
    + "_"
    + education["RANG_SOURCE_J10F"].astype("string")
)

education["CATEGORIE_ANALYSE"] = (
    education["TYPEQU_ANALYSE"].map(TYPES)
)

education["GROUPE_CODE"] = (
    education["TYPEQU_ANALYSE"].str[:2]
)

education["GROUPE_ANALYSE"] = (
    education["GROUPE_CODE"].map(GROUPES)
)

education["SOURCE_DONNEES"] = "INSEE_BPE_2025"
education["STATUT_ACTIVITE_ACTUELLE"] = "NON_VERIFIE"

assert education["ID_OBSERVATION"].is_unique
assert education["GROUPE_ANALYSE"].notna().all()

display(education["GROUPE_ANALYSE"].value_counts())

GROUPE_ANALYSE
ENSEIGNEMENT_PREMIER_DEGRE             6946
FORMATION_CONTINUE_APPRENTISSAGE       1804
ENSEIGNEMENT_COLLEGES                  1292
ENSEIGNEMENT_LYCEES_SECTIONS            979
SUPERIEUR_NON_UNIVERSITAIRE             853
SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE     583
SERVICES_UNIVERSITAIRES                 269
Name: count, dtype: int64

In [9]:
#Harmoniser les communes

education["CODGEO"] = education["CODGEO_SOURCE"]

arrondissements_paris = {
    f"751{numero:02d}" for numero in range(1, 21)
}

if codes_profil & arrondissements_paris:
    raise ValueError(
        "Le profil doit être communal, sans arrondissements parisiens."
    )

if "75056" in codes_profil:
    education.loc[
        education["CODGEO"].isin(arrondissements_paris),
        "CODGEO",
    ] = "75056"

if "93066" in codes_profil and "93059" not in codes_profil:
    education.loc[
        education["CODGEO"].eq("93059"),
        "CODGEO",
    ] = "93066"

hors_profil = education.loc[
    ~education["CODGEO"].isin(codes_profil)
].copy()

sauvegarder_csv(
    hors_profil,
    DOSSIER_INTERIM / "education_hors_profil.csv",
)

print("Observations hors profil :", len(hors_profil))

if not hors_profil.empty:
    display(
        hors_profil[
            ["CODGEO_SOURCE", "CODGEO", "TYPEQU_ANALYSE"]
        ].drop_duplicates()
    )

    raise ValueError(
        "Résoudre les communes non appariées avant de poursuivre."
    )

Observations hors profil : 0


In [10]:
#Préparer les noms, adresses

colonnes_facultatives = [
    "SIRET", "NOMRS", "CNOMRS",
    "NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR",
    "LATITUDE", "LONGITUDE",
    "CAPACITE", "INDIC_CAPA", "CL_PGE",
]

for colonne in colonnes_facultatives:
    if colonne not in education.columns:
        education[colonne] = pd.Series(
            pd.NA,
            index=education.index,
            dtype="string",
        )

education["NOM_AFFICHAGE"] = (
    nettoyer_texte(education["NOMRS"])
    .fillna(nettoyer_texte(education["CNOMRS"]))
)

parties_adresse = education[
    ["NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR"]
].apply(nettoyer_texte)

education["ADRESSE_RECOMPOSEE"] = (
    parties_adresse.fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

education["SIRET_NETTOYE"] = nettoyer_texte(education["SIRET"])

siret_conforme = (
    education["SIRET_NETTOYE"].str.fullmatch(r"\d{14}", na=False)
    & education["SIRET_NETTOYE"].ne("0" * 14).fillna(False)
)

education["SIRET_TYPE_REPETE"] = (
    siret_conforme
    & education.duplicated(
        subset=["SIRET_NETTOYE", "TYPEQU_ANALYSE"],
        keep=False,
    )
)

repetitions = education.loc[
    education["SIRET_TYPE_REPETE"]
].copy()

sauvegarder_csv(
    repetitions,
    DOSSIER_INTERIM / "siret_types_repetes_a_examiner.csv",
)

print("Observations avec SIRET et type répétés :", len(repetitions))
print("Aucune suppression automatique.")

Observations avec SIRET et type répétés : 97
Aucune suppression automatique.


In [11]:
#Contrôler les coordonnées

education["LATITUDE_NUM"] = convertir_nombre(
    education["LATITUDE"]
)

education["LONGITUDE_NUM"] = convertir_nombre(
    education["LONGITUDE"]
)

coordonnees_presentes = (
    education["LATITUDE_NUM"].notna()
    & education["LONGITUDE_NUM"].notna()
)

coordonnees_plausibles = (
    education["LATITUDE_NUM"].between(48.0, 49.3)
    & education["LONGITUDE_NUM"].between(1.4, 3.7)
).fillna(False)

education["STATUT_COORDONNEES"] = "ABSENTES_OU_NON_NUMERIQUES"

education.loc[
    coordonnees_presentes,
    "STATUT_COORDONNEES",
] = "HORS_RECTANGLE_CONTROLE"

education.loc[
    coordonnees_plausibles,
    "STATUT_COORDONNEES",
] = "PLAUSIBLES_A_VERIFIER"

controle_coordonnees = (
    education.groupby(
        ["GROUPE_ANALYSE", "STATUT_COORDONNEES"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    controle_coordonnees,
    DOSSIER_INTERIM / "controle_coordonnees.csv",
)

display(controle_coordonnees)

,GROUPE_ANALYSE,STATUT_COORDONNEES,NB_OBSERVATIONS
0,ENSEIGNEMENT_COLLEGES,PLAUSIBLES_A_VERIFIER,1292
1,ENSEIGNEMENT_LYCEES_SECTIONS,PLAUSIBLES_A_VERIFIER,979
2,ENSEIGNEMENT_PREMIER_DEGRE,ABSENTES_OU_NON_NUMERIQUES,1
3,ENSEIGNEMENT_PREMIER_DEGRE,PLAUSIBLES_A_VERIFIER,6945
4,FORMATION_CONTINUE_APPRENTISSAGE,ABSENTES_OU_NON_NUMERIQUES,3
5,FORMATION_CONTINUE_APPRENTISSAGE,PLAUSIBLES_A_VERIFIER,1801
6,SERVICES_UNIVERSITAIRES,PLAUSIBLES_A_VERIFIER,269
7,SUPERIEUR_NON_UNIVERSITAIRE,PLAUSIBLES_A_VERIFIER,853
8,SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE,ABSENTES_OU_NON_NUMERIQUES,1
9,SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE,PLAUSIBLES_A_VERIFIER,582


In [12]:
#Documenter les effectifs

CODES_EFFECTIFS_SCOLAIRES = [
    "C107", "C108", "C109",
    "C201", "C301", "C302", "C303",
]

PERIODES_EFFECTIFS = {
    **{
        code: "RENTREE_2024"
        for code in CODES_EFFECTIFS_SCOLAIRES
    },
    "C505": "RENTREE_2023",
    "C603": "ANNEE_CIVILE_2022",
}

capacite = convertir_nombre(education["CAPACITE"])

capacite_valide = (
    capacite.notna()
    & capacite.ge(0)
    & capacite.mod(1).eq(0)
).fillna(False)

concerne_capacite = nettoyer_texte(
    education["INDIC_CAPA"]
).eq("1").fillna(False)

champ_documente = education["TYPEQU_ANALYSE"].isin(
    PERIODES_EFFECTIFS
)

education["EFFECTIF_SOURCE_NUM"] = capacite.where(
    capacite_valide & concerne_capacite & champ_documente
)

education["PERIODE_EFFECTIF_SOURCE"] = (
    education["TYPEQU_ANALYSE"]
    .map(PERIODES_EFFECTIFS)
    .astype("string")
)

education["STATUT_EFFECTIF_SOURCE"] = "HORS_CHAMP_DOCUMENTE"

education.loc[
    champ_documente,
    "STATUT_EFFECTIF_SOURCE",
] = "MANQUANT_OU_NON_EXPLOITABLE"

education.loc[
    education["EFFECTIF_SOURCE_NUM"].notna(),
    "STATUT_EFFECTIF_SOURCE",
] = "RENSEIGNE"

controle_effectifs = (
    education.groupby(
        ["CATEGORIE_ANALYSE", "STATUT_EFFECTIF_SOURCE"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    controle_effectifs,
    DOSSIER_INTERIM / "controle_effectifs_source.csv",
)

display(controle_effectifs)

,CATEGORIE_ANALYSE,STATUT_EFFECTIF_SOURCE,NB_OBSERVATIONS
0,AUTRES_FORMATIONS_POSTBAC_NON_UNIVERSITAIRES,HORS_CHAMP_DOCUMENTE,378
1,AUTRES_STRUCTURES_ENSEIGNEMENT_SUPERIEUR,HORS_CHAMP_DOCUMENTE,134
2,COLLEGES,MANQUANT_OU_NON_EXPLOITABLE,22
3,COLLEGES,RENSEIGNE,1270
4,ECOLES_ELEMENTAIRES,MANQUANT_OU_NON_EXPLOITABLE,7
5,ECOLES_ELEMENTAIRES,RENSEIGNE,2702
6,ECOLES_FORMATIONS_SANITAIRES_SOCIALES,HORS_CHAMP_DOCUMENTE,240
7,ECOLES_INGENIEURS,HORS_CHAMP_DOCUMENTE,63
8,ECOLES_MATERNELLES,MANQUANT_OU_NON_EXPLOITABLE,8
9,ECOLES_MATERNELLES,RENSEIGNE,2764


In [13]:
#Préparer le diasgnostic des classes préparatoires

codes_lycees_cpge = ["C301", "C302", "C303"]

est_lycee_concerne = education["TYPEQU_ANALYSE"].isin(
    codes_lycees_cpge
)

education["STATUT_CPGE_LYCEE"] = "HORS_CHAMP"

education.loc[
    est_lycee_concerne,
    "STATUT_CPGE_LYCEE",
] = (
    nettoyer_texte(
        education.loc[est_lycee_concerne, "CL_PGE"]
    )
    .map({"0": "ABSENCE_DECLAREE", "1": "PRESENCE_DECLAREE"})
    .fillna("INCONNU")
)

diagnostic_postbac = education.loc[
    education["GROUPE_CODE"].isin(["C4", "C5"])
    | education["STATUT_CPGE_LYCEE"].eq("PRESENCE_DECLAREE")
].copy()

sauvegarder_csv(
    diagnostic_postbac,
    DOSSIER_INTERIM / "observations_postbac_a_examiner.csv",
)

print("Observations à examiner pour le postbac :", len(diagnostic_postbac))

Observations à examiner pour le postbac : 1554


In [14]:
#Compter par sous-catégorie et groupe

COLONNES_TYPES = {
    code: f"NB_OBS_{libelle}_BPE2025"
    for code, libelle in TYPES.items()
}

COLONNES_GROUPES = {
    code: f"NB_OBS_{libelle}_BPE2025"
    for code, libelle in GROUPES.items()
}

comptages = pd.crosstab(
    education["CODGEO"],
    education["TYPEQU_ANALYSE"],
)

comptages = (
    comptages.reindex(columns=list(TYPES), fill_value=0)
    .rename(columns=COLONNES_TYPES)
    .reset_index()
)

indicateurs = profil[["CODGEO"]].merge(
    comptages,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonnes_types = list(COLONNES_TYPES.values())

indicateurs[colonnes_types] = (
    indicateurs[colonnes_types]
    .fillna(0)
    .astype("int64")
)

for groupe, colonne_sortie in COLONNES_GROUPES.items():
    colonnes_a_sommer = [
        COLONNES_TYPES[code]
        for code in TYPES
        if code.startswith(groupe)
    ]

    indicateurs[colonne_sortie] = (
        indicateurs[colonnes_a_sommer].sum(axis=1)
    )

COLONNE_TOTAL = "NB_OBS_EDUCATION_BPE2025"

indicateurs[COLONNE_TOTAL] = (
    indicateurs[colonnes_types].sum(axis=1)
)

assert indicateurs[COLONNE_TOTAL].sum() == len(education)

assert (
    indicateurs[list(COLONNES_GROUPES.values())].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

print("Comptages cohérents.")

Comptages cohérents.


In [15]:
#Calculer les parts des groupes et intragroupe

total = indicateurs[COLONNE_TOTAL].replace(0, np.nan)

PARTS_GROUPES = {}
PARTS_INTRAGROUPE = {}
series_parts = {}

for code, libelle in GROUPES.items():
    nom_part = f"PART_{libelle}_J10F_PCT"

    series_parts[nom_part] = (
        indicateurs[COLONNES_GROUPES[code]]
        .div(total)
        .mul(100)
    )

    PARTS_GROUPES[code] = nom_part

for code, libelle in TYPES.items():
    nom_part = f"PART_{libelle}_DANS_GROUPE_PCT"

    total_groupe = indicateurs[
        COLONNES_GROUPES[code[:2]]
    ].replace(0, np.nan)

    series_parts[nom_part] = (
        indicateurs[COLONNES_TYPES[code]]
        .div(total_groupe)
        .mul(100)
    )

    PARTS_INTRAGROUPE[code] = nom_part

indicateurs = pd.concat(
    [
        indicateurs,
        pd.DataFrame(series_parts, index=indicateurs.index),
    ],
    axis=1,
)

In [16]:
#Contrôler les pourcentages

def verifier_parts(table, colonnes, colonne_total):
    avec_observations = table[colonne_total].gt(0)

    valeurs = table.loc[avec_observations, colonnes]

    assert valeurs.notna().all().all()
    assert ((valeurs >= 0) & (valeurs <= 100)).all().all()

    assert np.allclose(
        valeurs.sum(axis=1).to_numpy(dtype=float),
        100.0,
    )

    assert table.loc[
        ~avec_observations, colonnes
    ].isna().all().all()


verifier_parts(
    indicateurs,
    list(PARTS_GROUPES.values()),
    COLONNE_TOTAL,
)

for groupe in GROUPES:
    colonnes = [
        PARTS_INTRAGROUPE[code]
        for code in TYPES
        if code.startswith(groupe)
    ]

    verifier_parts(
        indicateurs,
        colonnes,
        COLONNES_GROUPES[groupe],
    )

print("Parts des groupes et parts intragroupe contrôlées.")

Parts des groupes et parts intragroupe contrôlées.


In [17]:
#Reprendre la population de référence

COLONNE_POPULATION = None
metadata_j10e = {}

if FICHIER_METADATA_J10E.is_file():
    with open(
        FICHIER_METADATA_J10E,
        "r",
        encoding="utf-8",
    ) as flux:
        metadata_j10e = json.load(flux)

if COLONNE_POPULATION is None:
    COLONNE_POPULATION = metadata_j10e.get("colonne_population")

if COLONNE_POPULATION is None:
    candidats = [
        colonne
        for colonne in [
            "POPULATION",
            "POPULATION_MUNICIPALE",
            "P23_POP",
            "P22_POP",
            "POP",
        ]
        if colonne in profil.columns
    ]

    if len(candidats) == 1:
        COLONNE_POPULATION = candidats[0]
    else:
        print("Candidats :", candidats)
        raise ValueError(
            "Renseigne COLONNE_POPULATION comme au J10e."
        )

if COLONNE_POPULATION not in profil.columns:
    raise ValueError("Colonne de population introuvable.")

population_par_code = (
    profil.set_index("CODGEO")[COLONNE_POPULATION]
    .pipe(convertir_nombre)
)

population = indicateurs["CODGEO"].map(population_par_code)

population_valide = population.gt(0).fillna(False)
denominateur = population.where(population_valide)

print("Population utilisée :", COLONNE_POPULATION)
print(
    "Communes sans population positive exploitable :",
    int((~population_valide).sum()),
)

Population utilisée : POPULATION_2022
Communes sans population positive exploitable : 0


In [18]:
#Calculer uniquement les densités de groupes

DENSITES = {
    COLONNES_GROUPES[code]:
        f"DENSITE_OBS_{libelle}_10000_HAB"
    for code, libelle in GROUPES.items()
}

series_densites = {
    colonne_densite: (
        indicateurs[colonne_nombre]
        .div(denominateur)
        .mul(10_000)
    )
    for colonne_nombre, colonne_densite in DENSITES.items()
}

indicateurs = pd.concat(
    [
        indicateurs,
        pd.DataFrame(series_densites, index=indicateurs.index),
    ],
    axis=1,
)

print("Nombre de densités :", len(DENSITES))
print(list(DENSITES.values()))

Nombre de densités : 7
['DENSITE_OBS_ENSEIGNEMENT_PREMIER_DEGRE_10000_HAB', 'DENSITE_OBS_ENSEIGNEMENT_COLLEGES_10000_HAB', 'DENSITE_OBS_ENSEIGNEMENT_LYCEES_SECTIONS_10000_HAB', 'DENSITE_OBS_SUPERIEUR_NON_UNIVERSITAIRE_10000_HAB', 'DENSITE_OBS_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_10000_HAB', 'DENSITE_OBS_FORMATION_CONTINUE_APPRENTISSAGE_10000_HAB', 'DENSITE_OBS_SERVICES_UNIVERSITAIRES_10000_HAB']


In [19]:
#Joindre au profil J10e

nouvelles_colonnes = set(indicateurs.columns) - {"CODGEO"}
collisions = nouvelles_colonnes & set(profil.columns)

if collisions:
    raise ValueError(
        "Colonnes déjà présentes dans le profil : "
        + ", ".join(sorted(collisions))
    )

if indicateurs.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans les indicateurs.")

profil_j10f = profil.merge(
    indicateurs,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert len(profil_j10f) == len(profil)
assert profil_j10f["CODGEO"].is_unique

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10f[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

print("Colonnes avant :", len(profil.columns))
print("Colonnes après :", len(profil_j10f.columns))

Colonnes avant : 871
Colonnes après : 945


In [20]:
#Examiner les résultats

colonne_nom = next(
    (
        colonne
        for colonne in ["LIBELLE", "NOM_COMMUNE", "LIBGEO", "NOM_COM"]
        if colonne in profil_j10f.columns
    ),
    None,
)

colonnes_affichage = ["CODGEO"]

if colonne_nom:
    colonnes_affichage.append(colonne_nom)

colonnes_affichage += [
    COLONNE_POPULATION,
    "NB_OBS_ENSEIGNEMENT_PREMIER_DEGRE_BPE2025",
    "NB_OBS_COLLEGES_BPE2025",
    "NB_OBS_ENSEIGNEMENT_LYCEES_SECTIONS_BPE2025",
    "NB_OBS_SUPERIEUR_NON_UNIVERSITAIRE_BPE2025",
    "NB_OBS_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_BPE2025",
    "NB_OBS_RESIDENCES_UNIVERSITAIRES_BPE2025",
    "NB_OBS_RESTAURANTS_UNIVERSITAIRES_BPE2025",
]

display(
    profil_j10f[colonnes_affichage]
    .sort_values(
        "NB_OBS_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_BPE2025",
        ascending=False,
    )
    .head(20)
)

print("Cergy et Pontoise :")

display(
    profil_j10f.loc[
        profil_j10f["CODGEO"].isin(["95127", "95500"]),
        colonnes_affichage,
    ]
)

bilan_categories = (
    education.groupby(
        ["TYPEQU_ANALYSE", "CATEGORIE_ANALYSE", "GROUPE_ANALYSE"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    bilan_categories,
    DOSSIER_INTERIM / "bilan_categories.csv",
)

display(bilan_categories)

,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_ENSEIGNEMENT_PREMIER_DEGRE_BPE2025,NB_OBS_COLLEGES_BPE2025,NB_OBS_ENSEIGNEMENT_LYCEES_SECTIONS_BPE2025,NB_OBS_SUPERIEUR_NON_UNIVERSITAIRE_BPE2025,NB_OBS_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_BPE2025,NB_OBS_RESIDENCES_UNIVERSITAIRES_BPE2025,NB_OBS_RESTAURANTS_UNIVERSITAIRES_BPE2025
0,75056,Paris,2113705.0,841,218,241,466,312,91,59
835,91228,Évry-Courcouronnes,66700.0,47,7,6,6,28,5,2
1027,93066,Saint-Denis,148907.0,87,15,12,9,19,2,2
1048,94028,Créteil,92859.0,47,11,11,5,18,3,2
79,77083,Champs-sur-Marne,26661.0,20,3,2,3,18,3,0
1119,95127,Cergy,69578.0,52,7,4,12,15,5,3
984,92050,Nanterre,98119.0,49,9,6,10,13,1,1
894,91471,Orsay,16655.0,11,3,2,2,11,4,1
1035,93079,Villetaneuse,12432.0,7,2,0,0,10,3,1
896,91477,Palaiseau,36067.0,22,4,5,1,8,1,2


Cergy et Pontoise :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_ENSEIGNEMENT_PREMIER_DEGRE_BPE2025,NB_OBS_COLLEGES_BPE2025,NB_OBS_ENSEIGNEMENT_LYCEES_SECTIONS_BPE2025,NB_OBS_SUPERIEUR_NON_UNIVERSITAIRE_BPE2025,NB_OBS_SUPERIEUR_UNIVERSITAIRE_ET_ASSIMILE_BPE2025,NB_OBS_RESIDENCES_UNIVERSITAIRES_BPE2025,NB_OBS_RESTAURANTS_UNIVERSITAIRES_BPE2025
1119,95127,Cergy,69578.0,52,7,4,12,15,5,3
1220,95500,Pontoise,31623.0,21,6,5,4,6,0,1


,TYPEQU_ANALYSE,CATEGORIE_ANALYSE,GROUPE_ANALYSE,NB_OBSERVATIONS
0,C107,ECOLES_MATERNELLES,ENSEIGNEMENT_PREMIER_DEGRE,2772
1,C108,ECOLES_MIXTES_MATERNELLE_ELEMENTAIRE,ENSEIGNEMENT_PREMIER_DEGRE,1465
2,C109,ECOLES_ELEMENTAIRES,ENSEIGNEMENT_PREMIER_DEGRE,2709
3,C201,COLLEGES,ENSEIGNEMENT_COLLEGES,1292
4,C301,LYCEES_GENERAUX_TECHNOLOGIQUES,ENSEIGNEMENT_LYCEES_SECTIONS,634
5,C302,LYCEES_PROFESSIONNELS,ENSEIGNEMENT_LYCEES_SECTIONS,138
6,C303,LYCEES_ET_ETABLISSEMENTS_SECONDAIRES_AGRICOLES,ENSEIGNEMENT_LYCEES_SECTIONS,20
7,C304,SECTIONS_GENERALES_TECHNOLOGIQUES,ENSEIGNEMENT_LYCEES_SECTIONS,9
8,C305,SECTIONS_PROFESSIONNELLES,ENSEIGNEMENT_LYCEES_SECTIONS,178
9,C401,FORMATIONS_STS_CPGE,SUPERIEUR_NON_UNIVERSITAIRE,58


In [21]:
#Préparer la future étude des sites étudiants

FICHIER_SITES_ETUDIANTS = (
    DOSSIER_INTERIM / "sites_etudiants_documentation.csv"
)

colonnes_sites = [
    "ID_SITE",
    "NOM_SITE",
    "ETABLISSEMENT",
    "CODGEO",
    "ADRESSE",
    "LATITUDE_ENTREE",
    "LONGITUDE_ENTREE",
    "ANNEE_UNIVERSITAIRE",
    "EFFECTIF_SITE",
    "PERIMETRE_EFFECTIF",
    "SOURCE_URL",
    "DATE_VERIFICATION",
    "COMMENTAIRE",
]

if not FICHIER_SITES_ETUDIANTS.exists():
    sauvegarder_csv(
        pd.DataFrame(columns=colonnes_sites),
        FICHIER_SITES_ETUDIANTS,
    )

print("Fichier de documentation :", FICHIER_SITES_ETUDIANTS)

Fichier de documentation : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10f\sites_etudiants_documentation.csv


In [22]:
#Créer le dictionnaire des indicateurs

lignes_dictionnaire = []

def documenter(colonne, description, unite, denominateur=""):
    lignes_dictionnaire.append({
        "COLONNE": colonne,
        "DESCRIPTION": description,
        "UNITE": unite,
        "DENOMINATEUR": denominateur,
    })


for code, libelle in TYPES.items():
    documenter(
        COLONNES_TYPES[code],
        f"Observations BPE du type {code} : {libelle}",
        "observations",
    )

    documenter(
        PARTS_INTRAGROUPE[code],
        f"Part du type {code} dans son groupe",
        "%",
        COLONNES_GROUPES[code[:2]],
    )


for code, libelle in GROUPES.items():
    colonne_nombre = COLONNES_GROUPES[code]

    documenter(
        colonne_nombre,
        f"Observations du groupe {libelle}",
        "observations",
    )

    documenter(
        PARTS_GROUPES[code],
        f"Part du groupe {libelle} dans J10f",
        "%",
        COLONNE_TOTAL,
    )

    documenter(
        DENSITES[colonne_nombre],
        f"Densité du groupe {libelle}",
        "observations pour 10 000 habitants",
        COLONNE_POPULATION,
    )


documenter(
    COLONNE_TOTAL,
    "Total des observations du domaine éducatif retenues",
    "observations",
)

dictionnaire = pd.DataFrame(lignes_dictionnaire)

assert dictionnaire["COLONNE"].is_unique

assert set(dictionnaire["COLONNE"]) == (
    set(indicateurs.columns) - {"CODGEO"}
)

sauvegarder_csv(
    dictionnaire,
    DOSSIER_INTERIM / "dictionnaire_indicateurs_j10f.csv",
)

In [23]:
#Contrôles finaux

colonnes_effectifs = [
    *COLONNES_TYPES.values(),
    *COLONNES_GROUPES.values(),
    COLONNE_TOTAL,
]

assert len(profil_j10f) == len(profil)
assert profil_j10f["CODGEO"].is_unique
assert set(profil_j10f["CODGEO"]) == codes_profil

assert education["ID_OBSERVATION"].is_unique
assert education["CODGEO"].isin(codes_profil).all()

assert indicateurs[colonnes_effectifs].notna().all().all()
assert indicateurs[colonnes_effectifs].ge(0).all().all()

assert indicateurs[COLONNE_TOTAL].sum() == len(education)

assert (
    indicateurs[colonnes_types].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

assert (
    indicateurs[list(COLONNES_GROUPES.values())].sum(axis=1)
    == indicateurs[COLONNE_TOTAL]
).all()

for colonne in DENSITES.values():
    valeurs = indicateurs[colonne].dropna().astype(float)

    assert np.isfinite(valeurs).all()
    assert valeurs.ge(0).all()

    assert indicateurs.loc[
        ~population_valide, colonne
    ].isna().all()

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10f[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

controle = pd.DataFrame([{
    "COMMUNES_ENTREE": len(profil),
    "COMMUNES_SORTIE": len(profil_j10f),
    "OBSERVATIONS_EDUCATION": len(education),
    "OBSERVATIONS_HORS_PROFIL": len(hors_profil),
    "OBSERVATIONS_SIRET_TYPE_REPETE": len(repetitions),
    "COMMUNES_SANS_POPULATION_VALIDE": int(
        (~population_valide).sum()
    ),
    "ETAT": (
        "CONTROLES_STRUCTURELS_OK"
        if repetitions.empty
        else "CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER"
    ),
}])

sauvegarder_csv(
    controle,
    DOSSIER_INTERIM / "controle_jointures.csv",
)

display(controle)

,COMMUNES_ENTREE,COMMUNES_SORTIE,OBSERVATIONS_EDUCATION,OBSERVATIONS_HORS_PROFIL,OBSERVATIONS_SIRET_TYPE_REPETE,COMMUNES_SANS_POPULATION_VALIDE,ETAT
0,1266,1266,12726,0,97,0,CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER


In [24]:
#Enregistrer les résultats

sauvegarder_csv(education, FICHIER_DETAIL)
sauvegarder_csv(indicateurs, FICHIER_COMMUNES)
sauvegarder_csv(profil_j10f, FICHIER_SORTIE)

metadonnees = {
    "notebook": "10f_education_idf.ipynb",
    "date_traitement_utc": DATE_TRAITEMENT,
    "source": "INSEE_BPE_2025",
    "url_nomenclature": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_liste_hierarchisee_TYPEQU.html"
    ),
    "url_dictionnaire": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_anonymisee_dictionnaire_variables.html"
    ),
    "fichier_bpe": str(FICHIER_BPE),
    "sha256_bpe": HASH_BPE,
    "profil_entree": str(FICHIER_PROFIL),
    "sha256_profil_entree": sha256_fichier(FICHIER_PROFIL),
    "profil_sortie": str(FICHIER_SORTIE),
    "colonne_population": COLONNE_POPULATION,
    "types": TYPES,
    "groupes": GROUPES,
    "nombre_observations": len(education),
    "nombre_communes": len(profil_j10f),
    "observations_siret_type_repete": len(repetitions),
    "deduplication_automatique": False,
    "parts": {
        "groupes": "Dans le total J10f",
        "sous_categories": "Dans leur groupe uniquement",
    },
    "densites": "Groupes uniquement, pour 10 000 habitants",
    "periodes_effectifs_source": PERIODES_EFFECTIFS,
    "limites": [
        "Observations BPE, pas de campus ou bâtiments uniques.",
        "Présence possible de sections et de composantes.",
        "Aucun total général d'élèves ou d'étudiants calculé.",
        "Effectifs disponibles conservés dans le détail avec leur période.",
        "Population résidente distincte de la population étudiant sur place.",
        "Résidences universitaires : périmètre limité de la BPE.",
        "Densités rapportées à la population totale.",
        "Crèches et conservatoires déjà traités dans les étapes précédentes.",
        "Coordonnées contrôlées par rectangle uniquement.",
    ],
}

with open(
    DOSSIER_INTERIM / "metadata_j10f.json",
    "w",
    encoding="utf-8",
) as flux:
    json.dump(
        metadonnees,
        flux,
        ensure_ascii=False,
        indent=2,
    )

print("Détail :", FICHIER_DETAIL)
print("Indicateurs communaux :", FICHIER_COMMUNES)
print("Profil enrichi :", FICHIER_SORTIE)
print("J10f enregistré.")

Détail : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\education_idf_j10f.csv
Indicateurs communaux : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10f\education_communes.csv
Profil enrichi : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10f.csv
J10f enregistré.
